In [19]:
# Cell 1: Setup
# Load the already cleaned hospital dataset so this notebook documents the final cleaning workflow on the cleaned input file.
# The raw CSV is not used here.
import pandas as pd

# Read the cleaned CSV file provided for the internship submission.
df = pd.read_csv('hospital_cleaned.csv')

# Show the first few rows to confirm the input file loaded correctly.
print('Initial cleaned data preview:')
df.head()

# Print the dataset dimensions so the evaluator can quickly see rows, columns, and total cells.
print(f'Rows: {df.shape[0]}')
print(f'Columns: {df.shape[1]}')
print(f'Total cells: {df.shape[0] * df.shape[1]}')

# Display column counts, non-null counts, and data types.
print('\nInitial cleaned dataframe info:')
df.info()

Initial cleaned data preview:
Rows: 10000
Columns: 54
Total cells: 540000

Initial cleaned dataframe info:
<class 'pandas.DataFrame'>
RangeIndex: 10000 entries, 0 to 9999
Data columns (total 54 columns):
 #   Column                          Non-Null Count  Dtype  
---  ------                          --------------  -----  
 0   Hospital Name                   10000 non-null  str    
 1   Hospital Type                   10000 non-null  str    
 2   City                            10000 non-null  str    
 3   State                           10000 non-null  str    
 4   Department                      10000 non-null  str    
 5   Department ID                   10000 non-null  str    
 6   Nurses                          10000 non-null  int64  
 7   Staff Count                     10000 non-null  int64  
 8   Gender                          10000 non-null  str    
 9   Age                             10000 non-null  int64  
 10  Admission Date                  10000 non-null  str    
 11

In [20]:
# Cell 2: Handling Missing Data
# Even though the file is already cleaned, we verify that no missing values remain and show the standard treatment logic.
missing_values = df.isnull().sum()
print('Missing values by column:')
missing_table = missing_values[missing_values > 0].sort_values(ascending=False)
print(missing_table)

# Remove fully empty rows and columns if any export artifact exists.
df.dropna(how='all', inplace=True)
df.dropna(axis=1, how='all', inplace=True)

# Fill numeric gaps with the median only if any are present.
numeric_columns = df.select_dtypes(include='number').columns
for column in numeric_columns:
    if df[column].isnull().any():
        df[column] = df[column].fillna(df[column].median())

# Fill text gaps with a clear placeholder only if any are present.
text_columns = df.select_dtypes(include=['object', 'string']).columns
for column in text_columns:
    if df[column].isnull().any():
        df[column] = df[column].fillna('Unknown')

# Confirm the dataframe is free of missing values.
print('\nTotal missing values after cleaning:')
print(int(df.isnull().sum().sum()))

Missing values by column:
Transfer_Date    8240
dtype: int64

Total missing values after cleaning:
0


In [21]:
# Cell 3: Removing Duplicates
# Check for duplicate records and remove them if necessary.
duplicate_count = df.duplicated().sum()
print(f'Duplicate rows before removal: {duplicate_count}')

# Remove duplicate rows while keeping the first occurrence.
df.drop_duplicates(inplace=True)

# Verify duplicates are cleared.
print(f'Duplicate rows after removal: {df.duplicated().sum()}')

Duplicate rows before removal: 0
Duplicate rows after removal: 0


In [22]:
# Cell 4: Data Formatting
# Enforce the mentor-approved schema exactly for the cleaned input file.
# Text fields stay as text, date fields become datetime, numeric fields stay numeric, and flags remain 0/1 integers.
df.columns = df.columns.str.strip()
df.columns = df.columns.str.replace('Re-admission', 'Readmission', regex=False)

# Columns that must be stored as text.
text_columns = [
    'Hospital ID', 'Hospital Name', 'Hospital Type', 'City', 'State', 'Department', 'Department ID', 'Doctor',
    'Patient ID', 'Patient Name', 'Gender', 'Diagnosis', 'Treatment', 'Medication', 'Admission Type',
    'Test Result', 'Blood Type', 'Bed Occupied', 'Equipment', 'Insurance Provider', 'Readmission', 'Equipment ID',
    'Equipment Number', 'Equipment Status', 'Transferred', 'Transfer_From_Department', 'Transfer_To_Department'
]

# Columns that must be stored as dates.
date_columns = ['Admission Date', 'Discharge Date', 'Transfer_Date']

# Columns that must be stored as numbers.
number_columns = [
    'Nurses', 'Staff Count', 'Age', 'Beds Available', 'ICU Beds', 'Bed Number', 'Length of Stay', 'Room No',
    'Billing Amount', 'Dept_Bed_Capacity_Derived', 'Beds_Occupied_Count', 'Equipment_Total_Inventory',
    'Equipment_Usage_Duration_Hours', 'Number_of_Transfers', 'Dept_ICU_Bed_Capacity_Derived',
    'Dept_Staff_Capacity_Derived'
]

# Columns that must be stored as percentage/derived numeric values.
percent_columns = [
    'Admissions_Rate_%_Derived', 'Staff_Utilization_%_Derived', 'Bed_Occupancy_Rate_%',
    'Bed_Occupancy_Rate_Calc', 'ICU_Occupancy_Rate_Calc', 'Staff_Utilization_Calc'
]

# Binary flag columns that must remain 0/1 integers.
flag_columns = ['Readmission_Flag', 'Transferred_Flag', 'Equipment_InUse_Flag']

# Preserve any extra columns in the file, but still keep them consistent if they exist.
for column in text_columns:
    if column in df.columns:
        df[column] = df[column].astype('string').str.strip()

for column in date_columns:
    if column in df.columns:
        df[column] = pd.to_datetime(df[column], format='%Y-%m-%d', errors='coerce')

for column in number_columns + percent_columns:
    if column in df.columns:
        df[column] = pd.to_numeric(df[column], errors='coerce')

for column in flag_columns:
    if column in df.columns:
        df[column] = pd.to_numeric(df[column], errors='coerce').fillna(0).astype('int64')

# Show the resulting dtype summary so the schema mapping is visible in the notebook output.
print(df.dtypes)

Hospital Name                            string
Hospital Type                            string
City                                     string
State                                    string
Department                               string
Department ID                            string
Nurses                                    int64
Staff Count                               int64
Gender                                   string
Age                                       int64
Admission Date                    datetime64[s]
Discharge Date                    datetime64[s]
Diagnosis                                string
Treatment                                string
Medication                               string
Admission Type                           string
Test Result                              string
Blood Type                               string
Beds Available                            int64
ICU Beds                                  int64
Bed Number                              

In [23]:
# Cell 5: Feature Selection
# Keep the schema-defined columns and remove only empty export artifacts.
columns_to_drop = ['Unnamed: 0', 'Unnamed: 62']
df.drop(columns=columns_to_drop, inplace=True, errors='ignore')

# Reorder columns to match the mentor-approved schema as closely as possible.
preferred_order = [
    'Hospital ID', 'Hospital Name', 'Hospital Type', 'City', 'State', 'Department', 'Department ID', 'Doctor',
    'Nurses', 'Staff Count', 'Patient ID', 'Patient Name', 'Gender', 'Age', 'Admission Date', 'Discharge Date',
    'Diagnosis', 'Treatment', 'Medication', 'Admission Type', 'Test Result', 'Blood Type', 'Beds Available',
    'ICU Beds', 'Bed Number', 'Bed Occupied', 'Equipment', 'Length of Stay', 'Room No', 'Billing Amount',
    'Insurance Provider', 'Readmission', 'Dept_Bed_Capacity_Derived', 'Beds_Occupied_Count', 'Equipment ID',
    'Equipment Number', 'Equipment Status', 'Equipment_Total_Inventory', 'Equipment_Usage_Duration_Hours',
    'Transferred', 'Transfer_From_Department', 'Transfer_To_Department', 'Transfer_Date', 'Number_of_Transfers',
    'Dept_ICU_Bed_Capacity_Derived', 'Dept_Staff_Capacity_Derived', 'Admissions_Rate_%_Derived',
    'Staff_Utilization_%_Derived', 'Bed_Occupancy_Rate_%', 'Bed_Occupancy_Rate_Calc', 'ICU_Occupancy_Rate_Calc',
    'Staff_Utilization_Calc', 'Readmission_Flag', 'Transferred_Flag', 'Equipment_InUse_Flag', 'Total_Admissions',
    'Overall_Occupancy_Rate', 'Average_Length_of_Stay', 'Readmission_Rate', 'Bed_Utilization_Rate'
]
existing_order = [column for column in preferred_order if column in df.columns]
remaining_columns = [column for column in df.columns if column not in existing_order]
df = df[existing_order + remaining_columns]

# Show the final columns that will be exported.
print('Columns retained after feature selection:')
print(df.columns.tolist())

Columns retained after feature selection:
['Hospital Name', 'Hospital Type', 'City', 'State', 'Department', 'Department ID', 'Nurses', 'Staff Count', 'Gender', 'Age', 'Admission Date', 'Discharge Date', 'Diagnosis', 'Treatment', 'Medication', 'Admission Type', 'Test Result', 'Blood Type', 'Beds Available', 'ICU Beds', 'Bed Number', 'Bed Occupied', 'Equipment', 'Length of Stay', 'Room No', 'Billing Amount', 'Insurance Provider', 'Readmission', 'Dept_Bed_Capacity_Derived', 'Beds_Occupied_Count', 'Equipment Status', 'Equipment_Total_Inventory', 'Equipment_Usage_Duration_Hours', 'Transferred', 'Transfer_From_Department', 'Transfer_To_Department', 'Transfer_Date', 'Number_of_Transfers', 'Dept_ICU_Bed_Capacity_Derived', 'Dept_Staff_Capacity_Derived', 'Admissions_Rate_%_Derived', 'Staff_Utilization_%_Derived', 'Bed_Occupancy_Rate_%', 'Bed_Occupancy_Rate_Calc', 'ICU_Occupancy_Rate_Calc', 'Staff_Utilization_Calc', 'Readmission_Flag', 'Transferred_Flag', 'Equipment_InUse_Flag', 'Total_Admissions

In [24]:
# Cell 6: Export
# Save the cleaned dataset to a CSV file

output_file = 'hospital_cleaned.csv'

try:
    df.to_csv(output_file, index=False)
except PermissionError:
    output_file = 'hospital_cleaned_export.csv'
    df.to_csv(output_file, index=False)

# Confirm the final output path.
print(f'Cleaned dataset saved successfully to: {output_file}')

Cleaned dataset saved successfully to: hospital_cleaned_export.csv
